# Persistent Sessions: Save, Resume & Fork

The SDK supports persistent sessions — conversations that can be saved by their `session_id`, resumed later, and forked into independent branches that continue on their own.


In [1]:
from pathlib import Path

from claude_agent_sdk import (
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)

PROJECT_ROOT = Path.cwd().parent  # this notebook lives in episodes/, project root is one level up
SESSION_FILE = PROJECT_ROOT / "session_id.txt"


def save_session_id(session_id: str) -> None:
    # Every session has a unique session_id. Saving it to a file is how we
    # "remember" which conversation to come back to later.
    SESSION_FILE.parent.mkdir(parents=True, exist_ok=True)
    SESSION_FILE.write_text(session_id)


def load_session_id() -> str:
    return SESSION_FILE.read_text().strip()

## Start a session and save its `session_id`


In [2]:
async def start_session() -> None:
    async with ClaudeSDKClient(options=ClaudeAgentOptions(model="haiku")) as client:
        await client.query("Remember this: my project's codename is 'Ironwood'. Acknowledge only.")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"[turn 1] {message.result}")
                # message.session_id: the SDK's ID for this exact conversation.
                # Save it now so we can come back to this same conversation later.
                save_session_id(message.session_id)


await start_session()
print(f"Saved session_id to {SESSION_FILE}")

[turn 1] Acknowledged. I'll remember that your project's codename is **Ironwood**. 🔨

How can I help you with Ironwood today?
Saved session_id to /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/session_id.txt


## Resume it — the context is still there


In [3]:
async def resume_session() -> str:
    # resume=<session_id>: tells the SDK "continue THIS exact conversation"
    # instead of starting a fresh one — Claude still remembers everything
    # said in that earlier session.
    options = ClaudeAgentOptions(model="haiku", resume=load_session_id())
    async with ClaudeSDKClient(options=options) as client:
        await client.query("What's my project's codename?")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"[resumed] {message.result}")
                return message.session_id
        return ""


resumed_session_id = await resume_session()

[resumed] Your project's codename is **Ironwood**.


## Fork into two independent continuations

Both forks start from the exact same point. One renames the codename inside its own branch; the other just asks what the codename is. If forking truly creates independent branches, the second fork should still see the _original_ codename — untouched by what happened in the first fork.


In [5]:
async def fork_and_continue(base_session_id: str, next_message: str) -> None:
    # fork_session=True: instead of continuing the ORIGINAL session, this
    # creates a brand-new, independent copy that starts from the same point.
    # Changes made in one fork never affect the original session or any
    # other fork made from it.
    options = ClaudeAgentOptions(model="haiku", resume=base_session_id, fork_session=True)
    async with ClaudeSDKClient(options=options) as client:
        await client.query(next_message)
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"[fork: {next_message!r}]\n{message.result}\n")


# Fork 1: rename the codename — but only inside this fork's own branch.
await fork_and_continue(
    resumed_session_id,
    "Change the codename to 'Bluejay' for this branch only. Acknowledge only.",
)
# Fork 2: made independently from the SAME original session — should still
# see the ORIGINAL codename, proving the two forks don't affect each other.
await fork_and_continue(
    resumed_session_id,
    "What is the codename? Just answer, don't change anything.",
)

[fork: "Change the codename to 'Bluejay' for this branch only. Acknowledge only."]
Acknowledged. I've updated the codename to **Bluejay** for this branch only.

[fork: "What is the codename? Just answer, don't change anything."]
Ironwood



In [6]:
# Plain Python cleanup — delete the scratch file we used to store the session id.
SESSION_FILE.unlink(missing_ok=True)
print(f"Deleted {SESSION_FILE}")

Deleted /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/session_id.txt
